# Session 2a — Downloading Barbados economic data

This notebook is the **live data-download** step of the Barbados nowcasting pipeline. It retrieves series from the Barbados Statistical Service (BSS), the Central Bank of Barbados (CBB), FRED, EIA, the US Census Bureau, Statistics Canada and the UK ONS, and writes one CSV per source into `../raw/`.

> **Note:** this session requires an internet connection (and a FRED API key). All outputs are already packaged in `../raw/`, so later sessions work even if a download fails today.

In [ ]:
'''
This part of the code loads the key libraries.
If there is an error here, it is most likely a library needs to be installed.
https://stats.gov.bb/statistics/publications/
'''

import io, os, re, urllib3, requests, datetime,  tabula, camelot, PyPDF2, glob
import pandas as pd
import numpy as np
from datetime import date
from bs4 import BeautifulSoup
from dateutil.parser import parse

#warnings.simplefilter("ignore", category=pd.errors.SettingWithCopyWarning)
import warnings
warnings.filterwarnings('ignore')

'''
Defines folders.
The variable "d" captures the current folder where all programs are saved.
/source/ is where raw data is stored
/data/ is where the clean dataset is stored
folder_spec saves downloaded files (pdfs)
'''
# =========================================================
# Paths
# =========================================================
# Define local project directories (relative to current working directory)
d = f"{os.getcwd()}/"
PATH_RAW = os.path.join(d, "..", "data")   # raw inputs (downloads / intermediate CSVs)
folder_spec = PATH_RAW

'''
When connecting to a website, it is a good idea to mask the connection.
hdr masks the connection. It informs the website that you are connecting with a Browser. This reduces errors.
'''
hdr = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537'}

'''
The extract_page_text function opens a pdf an reads all the text. It is used to search specific pages
'''
def extract_page_text(pdf_path):
    with open(pdf_path, 'rb') as pdf_file:
        reader = PyPDF2.PdfReader(pdf_file)
        page_texts = [page.extract_text() for page in reader.pages]
    return page_texts

'''
Connecting to USA FRED requires an API Key. This key allows access to automatically download series.
Request FRED API key here: https://fred.stlouisfed.org/docs/api/api_key.html
'''
api_key = "72304a91a13cc0ea6c610ae5845c7493"

c:\Users\guerr\anaconda3\envs\nwcst\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\guerr\anaconda3\envs\nwcst\Lib\site-packages\pypdf\_crypt_providers\_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


# GDP

## CBB Excel File

Cleaning data from an Excel file

In [3]:
## Revised real GDP from the CBB workbook (REAL GDP sheet)
# Rebuilds bb_gdp.csv entirely from 'Revised Data.xlsx' (2010-Q1 .. 2024-Q4).

raw = pd.read_excel(f'{PATH_RAW}/Revised Data.xlsx', sheet_name='REAL GDP', header=None)
raw

,0,1,2,3,4,5,6,7,8,9,...,71,72,73,74,75,76,77,78,79,80
0,Estimates of Real GDP (2023 Prices),Actual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Estimate,NaN,NaN,NaN,NaN
1,Quarterly (In Millions of Barbados Dollars),Mar 2010,Jun 2010,Sep 2010,Dec 2010,TOT,Mar 2011,Jun 2011,Sep 2011,Dec 2011,...,Mar 2024,Jun 2024,Sep 2024,Dec 2024,TOT,Mar 2025,Jun 2025,Sep 2025,Dec 2025,TOT
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Traded Sectors,550.469322,496.662273,479.746752,534.499582,2061.377928,554.392654,491.130948,478.666155,476.819875,...,585.080073,458.160494,429.940236,547.301899,2020.482702,596.823362,467.984345,465.497223,551.892805,2082.197735
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Sugar,16.760007,8.77166,0,0,25.531667,11.08204,23.375999,0,0,...,1.596656,9.750533,0,0,11.347188,3.268695,7.508043,0,0,10.776738
6,Non-Sugar Agriculture Fishing,69.644131,78.95468,45.275696,49.622418,243.496925,75.868704,54.525881,40.208588,44.293521,...,59.288667,56.100484,49.019975,52.816514,217.22564,57.917638,58.701779,66.287129,64.925254,247.8318
7,Agriculture,86.404138,87.72634,45.275696,49.622418,269.028592,86.950744,77.901879,40.208588,44.293521,...,60.885322,65.851017,49.019975,52.816514,228.572828,61.186333,66.209821,66.287129,64.925254,258.608538
8,Manufacturing,195.159092,198.094861,184.335805,208.890852,786.480611,194.98573,194.081276,192.003435,200.643621,...,196.43087,191.845595,182.407299,176.502114,747.185879,194.44613,191.035952,182.574331,178.115853,746.172265
9,Tourism,268.906092,210.841071,250.13525,275.986312,1005.868726,272.456181,219.147793,246.454133,231.882733,...,327.763881,200.463883,198.512961,317.98327,1044.723995,341.190899,210.738571,216.635764,308.851698,1077.416931


In [ ]:
# Row 1 = headers: col 0 is the sector label, the rest are the quarter dates
# ('Mar 2010', ...) interleaved with annual 'TOT' columns, which we drop.
dates     = pd.to_datetime(raw.iloc[1, 1:], format='%b %Y', errors='coerce')
date_cols = dates.dropna().index

date_cols

Index([ 1,  2,  3,  4,  6,  7,  8,  9, 11, 12, 13, 14, 16, 17, 18, 19, 21, 22,
       23, 24, 26, 27, 28, 29, 31, 32, 33, 34, 36, 37, 38, 39, 41, 42, 43, 44,
       46, 47, 48, 49, 51, 52, 53, 54, 56, 57, 58, 59, 61, 62, 63, 64, 66, 67,
       68, 69, 71, 72, 73, 74, 76, 77, 78, 79],
      dtype='int64')

In [ ]:
# Map each sector row to the bb_gdp.csv column name. The revised sheet merges
# two pairs the old file kept apart, so the combined value goes where the old
# file stored it; the unused halves (non-sugar agriculture, transportation
# storage) stay empty, exactly as in the previous file.
sector_map = {
    'Traded Sectors'               : 'gdp_traded_sector',
    'Sugar'                        : 'gdp_sugar',
    'Non-Sugar Agriculture Fishing': 'gdp_fishing',
    'Manufacturing'                : 'gdp_manufacturing',
    'Tourism'                      : 'gdp_tourism',
    'Non-traded Sectors'           : 'gdp_non-traded_sector',
    'Mining & Quarrying'           : 'gdp_mining_quarrying',
    'Electricity, Gas & Water'     : 'gdp_electricity_gas_water',
    'Construction'                 : 'gdp_construction',
    'Wholesale & Retail'           : 'gdp_wholesale_retail',
    'Government'                   : 'gdp_government',
    'Communications'               : 'gdp_communications',
    'Business & Other Services'    : 'gdp_business_other_services',
    'Total'                        : 'gdp',
}

labels = raw[0].astype(str).str.strip()
rows   = labels[labels.isin(sector_map)].index
rows

Index([3, 5, 6, 8, 9, 12, 14, 15, 16, 17, 18, 20, 21, 24], dtype='int64')

In [7]:
df = raw.loc[rows, date_cols].T
df

,3,5,6,8,9,12,14,15,16,17,18,20,21,24
1,550.469322,16.760007,69.644131,195.159092,268.906092,2657.350951,13.344537,71.002746,186.692159,460.82679,396.71449,426.761337,1102.008891,3207.820273
2,496.662273,8.77166,78.95468,198.094861,210.841071,2483.892186,10.01379,70.365708,189.465012,455.676491,314.390144,380.9571,1063.023942,2980.554459
3,479.746752,0,45.275696,184.335805,250.13525,2571.01441,9.072809,72.139723,204.172866,457.74085,342.381254,396.07083,1089.436079,3050.761161
4,534.499582,0,49.622418,208.890852,275.986312,2506.866625,11.603699,68.939456,204.606782,511.383995,331.49042,393.912509,984.929764,3041.366207
6,554.392654,11.08204,75.868704,194.98573,272.456181,2599.59577,9.042038,66.480446,180.081473,506.950714,357.256653,417.129917,1062.654529,3153.988424
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,547.301899,0,52.816514,176.502114,317.98327,2926.247319,15.504339,84.989918,206.214969,766.547907,318.981652,392.624517,1141.384017,3473.549218
76,596.823362,3.268695,57.917638,194.44613,341.190899,2971.915325,13.524252,79.342891,183.792403,757.160322,341.08471,403.393206,1193.617541,3568.738687
77,467.984345,7.508043,58.701779,191.035952,210.738571,2824.167527,16.352097,82.039386,202.149805,706.497726,308.374862,338.072444,1170.681207,3292.151872
78,465.497223,0,66.287129,182.574331,216.635764,2828.766388,14.955337,82.709399,188.184418,709.309666,313.614469,343.182535,1176.810564,3294.263611


In [8]:
df.columns    = labels[rows].map(sector_map).values
df.index      = dates[date_cols].values
df.index.name = 'date'
df = df.apply(pd.to_numeric, errors='coerce').sort_index()
df

,gdp_traded_sector,gdp_sugar,gdp_fishing,gdp_manufacturing,gdp_tourism,gdp_non-traded_sector,gdp_mining_quarrying,gdp_electricity_gas_water,gdp_construction,gdp_wholesale_retail,gdp_government,gdp_communications,gdp_business_other_services,gdp
date,,,,,,,,,,,,,,
2010-03-01,550.469322,16.760007,69.644131,195.159092,268.906092,2657.350951,13.344537,71.002746,186.692159,460.826790,396.714490,426.761337,1102.008891,3207.820273
2010-06-01,496.662273,8.771660,78.954680,198.094861,210.841071,2483.892186,10.013790,70.365708,189.465012,455.676491,314.390144,380.957100,1063.023942,2980.554459
2010-09-01,479.746752,0.000000,45.275696,184.335805,250.135250,2571.014410,9.072809,72.139723,204.172866,457.740850,342.381254,396.070830,1089.436079,3050.761161
2010-12-01,534.499582,0.000000,49.622418,208.890852,275.986312,2506.866625,11.603699,68.939456,204.606782,511.383995,331.490420,393.912509,984.929764,3041.366207
2011-03-01,554.392654,11.082040,75.868704,194.985730,272.456181,2599.595770,9.042038,66.480446,180.081473,506.950714,357.256653,417.129917,1062.654529,3153.988424
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-01,547.301899,0.000000,52.816514,176.502114,317.983270,2926.247319,15.504339,84.989918,206.214969,766.547907,318.981652,392.624517,1141.384017,3473.549218
2025-03-01,596.823362,3.268695,57.917638,194.446130,341.190899,2971.915325,13.524252,79.342891,183.792403,757.160322,341.084710,403.393206,1193.617541,3568.738687
2025-06-01,467.984345,7.508043,58.701779,191.035952,210.738571,2824.167527,16.352097,82.039386,202.149805,706.497726,308.374862,338.072444,1170.681207,3292.151872


In [9]:
# Keep the exact columns and order of the existing file.
cols = ['gdp_business_other_services', 'gdp_communications', 'gdp_construction',
        'gdp_electricity_gas_water', 'gdp_fishing', 'gdp_government', 'gdp_manufacturing',
        'gdp_mining_quarrying', 'gdp_non-sugar_agriculture', 'gdp_non-traded_sector',
        'gdp_sugar', 'gdp', 'gdp_tourism', 'gdp_traded_sector', 'gdp_transportation_storage',
        'gdp_wholesale_retail']
df = df.reindex(columns=cols)

df.tail()

,gdp_business_other_services,gdp_communications,gdp_construction,gdp_electricity_gas_water,gdp_fishing,gdp_government,gdp_manufacturing,gdp_mining_quarrying,gdp_non-sugar_agriculture,gdp_non-traded_sector,gdp_sugar,gdp,gdp_tourism,gdp_traded_sector,gdp_transportation_storage,gdp_wholesale_retail
date,,,,,,,,,,,,,,,,
2024-12-01,1141.384017,392.624517,206.214969,84.989918,52.816514,318.981652,176.502114,15.504339,NaN,2926.247319,0.000000,3473.549218,317.983270,547.301899,NaN,766.547907
2025-03-01,1193.617541,403.393206,183.792403,79.342891,57.917638,341.084710,194.446130,13.524252,NaN,2971.915325,3.268695,3568.738687,341.190899,596.823362,NaN,757.160322
2025-06-01,1170.681207,338.072444,202.149805,82.039386,58.701779,308.374862,191.035952,16.352097,NaN,2824.167527,7.508043,3292.151872,210.738571,467.984345,NaN,706.497726
2025-09-01,1176.810564,343.182535,188.184418,82.709399,66.287129,313.614469,182.574331,14.955337,NaN,2828.766388,0.000000,3294.263611,216.635764,465.497223,NaN,709.309666
2025-12-01,1203.190722,401.902234,223.350426,86.581902,64.925254,318.981652,178.115853,15.922974,NaN,3029.601609,0.000000,3581.494414,308.851698,551.892805,NaN,779.671699


## FRED

In [18]:
series = ["SP500", "DJIA", "NASDAQCOM"]

df = pd.DataFrame()
for serie in series :
    #&realtime_end=9999-12-31
    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={serie}&api_key={api_key}&file_type=json"
    response = requests.get(url)
    response = response.json()
    dfi = pd.DataFrame(response['observations'])
    dfi = dfi[['value', 'date']]
    dfi.set_index('date', inplace=True)
    dfi.index = pd.to_datetime(dfi.index)
    dfi.rename(columns = { "value" : f"fred_{serie}" }, inplace=True)
    dfi[f"fred_{serie}"] = dfi[f"fred_{serie}"].apply(pd.to_numeric, errors='coerce')
    df = df.merge(dfi, left_index=True, right_index=True, how='outer')
    df = pd.DataFrame(df.resample("D").mean())
    #df = pd.concat([df, dfi], axis=1)
    df.sort_index(inplace=True)
    
df.to_csv(f"{PATH_RAW}/fred_data.csv")
print(f"FRED Updated: {df.index.max()}")
df.tail(5)

FRED Updated: 2026-06-08 00:00:00


,fred_SP500,fred_DJIA,fred_NASDAQCOM
date,,,
2026-06-04,7584.31,51561.93,26830.96
2026-06-05,7383.74,50866.78,25709.43
2026-06-06,NaN,NaN,NaN
2026-06-07,NaN,NaN,NaN
2026-06-08,7405.73,50786.01,25929.66


## EIA

Target: the EIA website

In [19]:
from io import BytesIO
'''
Access the EIA and download the xls file that has all data.
'''
url = "https://www.eia.gov/dnav/pet/hist/LeafHandler.ashx?n=PET&s=RBRTE&f=D"
url

'https://www.eia.gov/dnav/pet/hist/LeafHandler.ashx?n=PET&s=RBRTE&f=D'

First, we make a get request

In [20]:
r = requests.get(url=url, verify=None).content
r

b"<!DOCTYPE HTML PUBLIC '-//W3C//DTD HTML 4.01 Transitional//EN'>\r <html>\r <head>\r <meta http-equiv='X-UA-Compatible' content='IE=9' />\r <title>Europe Brent Spot Price FOB  (Dollars per Barrel)</title>\r <link rel='StyleSheet' href='../Styles/Pet_wrapper3.css' TYPE='text/css'>\r <link rel='StyleSheet' href='../Styles/leaf_new2.css' TYPE='text/css'>\r <link rel='StyleSheet' href='/styles/Eia_sitewideF.css' type='text/css' />\r <!-- Header Script -->\r <script language='JavaScript' src='/styles/eia_header.js' type='text/javascript'></script>\r <!--/ Header Script -->\r <script src='/global/includes/dnavs/leaf_handler.cfm' type='text/javascript'></script>\r <!-- Footer Script -->\r <script language='JavaScript' src='/styles/eia_footer.js' type='text/javascript'></script>\r <!--/ Footer Script -->\r </head>\r <body>\r <script language='JavaScript' type='text/javascript'>InsertEIAHeaderCode();</script>\r <table width='675' border='0' cellspacing='0' cellpadding='0'>\r <tr>\r <td class =

We use beautiful soup to read html language

In [21]:
soup = BeautifulSoup(r)
soup

<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN">
<html> <head> <meta content="IE=9" http-equiv="X-UA-Compatible"/> <title>Europe Brent Spot Price FOB  (Dollars per Barrel)</title> <link href="../Styles/Pet_wrapper3.css" rel="StyleSheet" type="text/css"/> <link href="../Styles/leaf_new2.css" rel="StyleSheet" type="text/css"/> <link href="/styles/Eia_sitewideF.css" rel="StyleSheet" type="text/css"/> <!-- Header Script --> <script language="JavaScript" src="/styles/eia_header.js" type="text/javascript"></script> <!--/ Header Script --> <script src="/global/includes/dnavs/leaf_handler.cfm" type="text/javascript"></script> <!-- Footer Script --> <script language="JavaScript" src="/styles/eia_footer.js" type="text/javascript"></script> <!--/ Footer Script --> </head> <body> <script language="JavaScript" type="text/javascript">InsertEIAHeaderCode();</script> <table border="0" cellpadding="0" cellspacing="0" width="675"> <tr> <td class="K"> </td> </tr> <tr> <td height="12"></td>

Create a list that filters all populated links (*a* tags with *href* link)

In [22]:
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
links

['rbrteD.htm',
 'rbrteD.htm',
 'rbrteW.htm',
 'rbrteW.htm',
 'rbrteM.htm',
 'rbrteM.htm',
 'rbrteA.htm',
 'rbrteA.htm',
 '../hist_xls/RBRTEd.xls',
 '../PET_PRI_SPT_S1_D.htm']

Get only the xls file

In [23]:
link = [ link for link in links if "xls" in link ][0].replace("../","")
link

'hist_xls/RBRTEd.xls'

Now make a get request to this file and open in pandas

In [24]:
get = f"https://www.eia.gov/dnav/pet/{link}"
df = pd.ExcelFile(get)

df = pd.read_excel( df , sheet_name=[ sheet for sheet in df.sheet_names if "data" in sheet.lower() ][0], skiprows=2)
df

,Date,Europe Brent Spot Price FOB (Dollars per Barrel)
0,1987-05-20,18.63
1,1987-05-21,18.45
2,1987-05-22,18.55
3,1987-05-25,18.60
4,1987-05-26,18.63
...,...,...
9897,2026-05-26,102.75
9898,2026-05-27,97.11
9899,2026-05-28,95.47
9900,2026-05-29,92.88


Now some simple data cleaning by renaming columns

In [25]:
df.columns = df.columns.str.lower()
df.set_index('date', inplace=True)
df = df.rename(columns=lambda x: "brent price" if "brent" in x else x)
df.index = pd.to_datetime(df.index)

# Save:
# df.to_csv(f"{PATH_RAW}/eia_brent.csv")
print(f"EIA  Updated: {df.index.max()}")

EIA  Updated: 2026-06-01 00:00:00


# BRB Central Bank Trade Statistics

## Central-Bank Trade in Goods Barbados

We are going to access the Central Bank trade-in Goods data page

In [26]:
url = "https://www.centralbank.org.bb/news/trade-in-goods-barbados"
url

'https://www.centralbank.org.bb/news/trade-in-goods-barbados'

The page has this format:

1. The page updates with blocks. You need to click "Read more" to access data
2. When you click an update, there is an excel available

First, list the links and get the latest one

In [27]:
r = requests.get(url=url, verify=None, headers=hdr).content

soup = BeautifulSoup(r)
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
links

['#',
 'Microsoft-edge:https://go.microsoft.com/fwlink/?linkid=2156983&pc=EE05&form=MY01TE&OCID=MY01TE',
 'https://www.centralbank.org.bb',
 'https://www.centralbank.org.bb',
 'javascript:;',
 'https://www.centralbank.org.bb/about-cbb',
 'https://www.centralbank.org.bb/history-of-the-bank',
 'https://www.centralbank.org.bb/vision-and-values',
 'https://www.centralbank.org.bb/role-of-the-bank',
 'https://www.centralbank.org.bb/objectives-and-functions',
 'https://www.centralbank.org.bb/leadership',
 'https://www.centralbank.org.bb/leadership/governor',
 'https://www.centralbank.org.bb/leadership/board-of-directors',
 'https://www.centralbank.org.bb/leadership/senior-officers',
 'https://www.centralbank.org.bb/leadership/past-governors',
 'https://www.centralbank.org.bb/corporate-social-responsibility',
 'https://www.centralbank.org.bb/corporate-social-responsibility/support-for-the-arts',
 'https://www.centralbank.org.bb/corporate-social-responsibility/support-for-education',
 'https://

In [28]:
links = [ link for link in links if "trade-in" in link and "g1-g4" in link  ]
links

['https://www.centralbank.org.bb/news/statistics-1/trade-in-goods-december-2025-g1-g4',
 'https://www.centralbank.org.bb/news/statistics-1/trade-in-goods-october-2025-g1-g4',
 'https://www.centralbank.org.bb/news/trade-in-goods-barbados/trade-in-goods-may-2025-g1-g4',
 'https://www.centralbank.org.bb/news/trade-in-goods-barbados/trade-in-goods-september-2024-g1-g4',
 'https://www.centralbank.org.bb/news/trade-in-goods-barbados/trade-in-goods-july-2024-g1-g4',
 'https://www.centralbank.org.bb/news/trade-in-goods-barbados/trade-in-goods-may-2024-g1-g4',
 'https://www.centralbank.org.bb/news/trade-in-goods-barbados/trade-in-goods-march-2024-g1-g4',
 'https://www.centralbank.org.bb/news/trade-in-goods-barbados/trade-in-goods-december-2023-g1-g4',
 'https://www.centralbank.org.bb/news/trade-in-goods-barbados/trade-in-goods-august-2023-g1-g4',
 'https://www.centralbank.org.bb/news/trade-in-goods-barbados/trade-in-goods-july-2023-g1-g4',
 'https://www.centralbank.org.bb/news/trade-in-goods-ba

The latest update page:

In [29]:
url = links[0]
url

'https://www.centralbank.org.bb/news/statistics-1/trade-in-goods-december-2025-g1-g4'

Now open the update and get access to the Excel

In [33]:
r = requests.get(url=url, verify=None, headers=hdr).content
soup = BeautifulSoup(r)
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
links = [ link for link in links if "xlsx" in link ]

get = links[0]
get

'https://cdn.centralbank.org.bb/documents/2026-03-06-09-38-02-G1G4TradeinGoodsDecember2025.xlsx'

Open the Excel

In [34]:
content = requests.get(url=get, verify=False, headers=hdr).content
df = pd.ExcelFile(io.BytesIO(content))
df

List the sheets:

In [35]:
df.sheet_names

['G1', 'G1A', 'G2A', 'G2B', 'G3A', 'G3B', 'G4A', 'G4B']

**Open and clean one sheet**

In [41]:
sheet = [ sheet for sheet in df.sheet_names if  "g1" in sheet.lower() ][0]

df0 = pd.read_excel(df, sheet_name=sheet, skiprows=4, header=[0,1], skipfooter=4)
df0.head(5)

Unnamed: 0_level_0 Unnamed: 1_level_0      Imports (CIF) EXPORTS (FOB)  \
  Unnamed: 0_level_1 Unnamed: 1_level_1 Unnamed: 2_level_1         Total   
0                NaN                NaT                G3A           NaN   
1                NaN                NaT                NaN           NaN   
2                NaN         1978-01-01         48860.9531    24242.5000   
3                NaN         1978-02-01         51867.2148    12886.3999   
4                NaN         1978-03-01         44471.5156    25830.0000   

                      Balance of Visible Trade  
  Domestic Re-Exports       Unnamed: 6_level_1  
0       G2        G4A                      NaN  
1      NaN        NaN                      NaN  
2     6610    17632.5              -24618.4531  
3     8474  4412.3999              -38980.8149  
4    21802       4028              -18641.5156

Rename columns

In [42]:
df0.columns = [' '.join(col).strip() for col in df0.columns.values]

df0 = df0.rename(columns = {
    'Unnamed: 1_level_0 Unnamed: 1_level_1' : 'date',
    'Imports (CIF) Unnamed: 2_level_1' : 'imp' ,
    'EXPORTS (FOB) Total' : 'exp_total',
    'EXPORTS (FOB) Domestic' : 'exp_dom',
    'EXPORTS (FOB) Re-Exports' : 'exp_re',
    'Retained Imports Unnamed: 6_level_1' : 'imp_re',
    'Balance of Visible Trade Unnamed: 7_level_1' : 'bal_trade',
})

df0.head(5)

,Unnamed: 0_level_0 Unnamed: 0_level_1,date,imp,exp_total,exp_dom,exp_re,Balance of Visible Trade Unnamed: 6_level_1
0,NaN,NaT,G3A,NaN,G2,G4A,NaN
1,NaN,NaT,NaN,NaN,NaN,NaN,NaN
2,NaN,1978-01-01,48860.9531,24242.5000,6610,17632.5,-24618.4531
3,NaN,1978-02-01,51867.2148,12886.3999,8474,4412.3999,-38980.8149
4,NaN,1978-03-01,44471.5156,25830.0000,21802,4028,-18641.5156


Drop columns with only missing values

In [43]:
df0 = df0.dropna(how='all', axis=1)
df0 = df0.dropna(how='all', subset = 'date', axis=0)
df0.head(5)

,date,imp,exp_total,exp_dom,exp_re,Balance of Visible Trade Unnamed: 6_level_1
2,1978-01-01,48860.9531,24242.5000,6610,17632.5,-24618.4531
3,1978-02-01,51867.2148,12886.3999,8474,4412.3999,-38980.8149
4,1978-03-01,44471.5156,25830.0000,21802,4028,-18641.5156
5,1978-04-01,48080.0352,19996.2000,17526,2470.2,-28083.8352
6,1978-05-01,57047.4766,16635.5000,12898,3737.5,-40411.9766


Basic reorganizing

In [ ]:
df0 = df0[['date', 'imp', 'exp_total', 'exp_dom', 'exp_re']]
df0['date'] = pd.to_datetime(df0['date'])
df0.set_index('date', inplace=True)
df0.head(5)

,imp,exp_total,exp_dom,exp_re
date,,,,
1978-01-01,48860.9531,24242.5000,6610,17632.5
1978-02-01,51867.2148,12886.3999,8474,4412.3999
1978-03-01,44471.5156,25830.0000,21802,4028
1978-04-01,48080.0352,19996.2000,17526,2470.2
1978-05-01,57047.4766,16635.5000,12898,3737.5


Creating next sports is quite easy:

In [45]:
df0['net_exp'] = df0['exp_total'] - df0['imp']
df0.tail(5)

,imp,exp_total,exp_dom,exp_re,net_exp
date,,,,,
2025-08-01,388010.63,57396.037,33269.382,24126.655,-330614.593
2025-09-01,380518.553,57143.699,35971.189,21172.51,-323374.854
2025-10-01,394926.711,66682.970,39353.606,27329.364,-328243.741
2025-11-01,347527.485,78452.038,41747.713,36704.325,-269075.447
2025-12-01,393455,72549.820,32029.67,40520.15,-320905.18


Some additional cleaning:

In [46]:
# Apply pd.to_numeric to all columns except the index
df0 = df0.apply(pd.to_numeric, errors='coerce')

# Drop all unnamed columns
df0 = df0.loc[:, ~df0.columns.str.contains('^unnamed')]
df0.dropna(how='all', axis=1, inplace=True)
df0.dropna(how='all', axis=0, inplace=True)

print(f"Range: {df0.index.min()}")
print(f"Updated: {df0.index.max()}")
df0.tail(5)

Range: 1978-01-01 00:00:00
Updated: 2025-12-01 00:00:00


,imp,exp_total,exp_dom,exp_re,net_exp
date,,,,,
2025-08-01,388010.630,57396.037,33269.382,24126.655,-330614.593
2025-09-01,380518.553,57143.699,35971.189,21172.510,-323374.854
2025-10-01,394926.711,66682.970,39353.606,27329.364,-328243.741
2025-11-01,347527.485,78452.038,41747.713,36704.325,-269075.447
2025-12-01,393455.000,72549.820,32029.670,40520.150,-320905.180


# Financial soundness

In [47]:
# Historical
url = "https://www.centralbank.org.bb/news/financial-soundness-indicators"
r = requests.get(url=url, verify=None, headers=hdr).content

soup = BeautifulSoup(r)
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
links = [ link for link in links if "core-" in link ]

url = links[0]
# -------------------
# Get latest page
r = requests.get(url=url, verify=None, headers=hdr).content

soup = BeautifulSoup(r)
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
links = [ link for link in links if "xlsx" in link ]

# ------------------
# get excel
get = links[0]
content = requests.get(url=get, verify=False, headers=hdr).content
#df = pd.ExcelFile(io.BytesIO(content))
df = pd.read_excel( io.BytesIO(content) , skiprows=5)
#df = pd.read_excel(get, skiprows=5)

# column names
df.columns = df.columns.astype(str).str.lower().str.strip()
df.columns = [re.sub(r'unnamed: \d+_level_\d+', '', str(col), flags=re.IGNORECASE).strip() for col in df.columns]
df.columns = [re.sub(r'\W+', '_', col).strip('_') for col in df.columns]
df.columns = [re.sub(r'\W+', '_', re.sub(r'\d+', '', str(col))).strip('_') for col in df.columns]
df.columns = df.columns.astype(str).str.lower().str.strip()
df = df.dropna(how='all', axis=0)
df = df.dropna(how='all', axis=1)

# dates management
quarter_to_month = {'1': 3, '2': 6, '3': 9, '4': 12}
df['period'] = pd.to_datetime([
    f"{p[-4:]}-{quarter_to_month[p[0]]:02d}-01" for p in df['period']
])
df.rename(columns = {'period' : 'date'} , inplace=True)
df.set_index('date', inplace=True)

# -
print(f'Last updated: {df.index.max()}')
df.to_csv(f'{PATH_RAW}/cbbb_finsound.csv')
df.tail(5)

Last updated: 2025-12-01 00:00:00


,regulatory_capital_to_risk_weighted_assets,tier__capital_to_risk_weighted_assets,nonperforming_loans_net_of_provisions_to_capital,tier__capital_to_assets,nonperforming_loans_to_total_gross_loans,loan_concentration_by_economic_activity,provisions_to_nonperforming_loans,return_on_assets,return_on_equity,interest_margin_to_gross_income,noninterest_expenses_to_gross_income,liquid_assets_to_total_assets,liquid_assets_to_short_term_liabilities,net_open_position_in_foreign_exchange_to_capital
date,,,,,,,,,,,,,,
2024-12-01,21.058251,19.758545,10.338130,12.583749,3.986409,65.839591,39.933745,1.440491,10.801969,64.535667,75.859169,28.649006,31.852428,20.475106
2025-03-01,19.570463,18.290433,10.441422,11.662563,3.760471,66.887050,39.499766,1.426932,10.113579,63.518509,75.117021,30.715797,41.834592,26.988446
2025-06-01,19.291021,17.986314,10.056882,11.572022,3.704287,66.393614,40.673102,1.344326,9.652657,63.886048,75.416255,30.135861,42.058966,23.552486
2025-09-01,19.118851,17.855752,10.230373,11.604987,3.582148,66.451679,38.760674,1.889940,13.481354,64.653015,76.409992,29.053508,40.772047,17.523150
2025-12-01,19.118348,18.165951,10.045738,11.855583,3.479991,65.720522,38.065031,2.451366,17.634134,64.737816,76.022583,28.810735,39.914391,22.079968


# Central Bank Fiscal data

## Historic data

Summary of Government Operations (Table F1)
Government Operations – Revenues and Grants (Table F2)
Government Operations – Expenditure and Net Lending (Table F3)

In [48]:
url = "https://www.centralbank.org.bb/news/summary-of-government-operations"
print(f"Retrieving historical: {url}")
r = requests.get(url=url, verify=None, headers=hdr).content

soup = BeautifulSoup(r)
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
links = [ link for link in links if ".xls" in link ]
if len(links)>0 :
    get = links[0]
    content = requests.get(url=get, verify=False, headers=hdr).content
    
    df = pd.ExcelFile(io.BytesIO(content))

    # Summary of Government Operations (Table F1)
    dfi = pd.read_excel(df, skiprows=4, sheet_name=df.sheet_names[0])
    
    # column names
    dfi.columns = dfi.columns.astype(str).str.lower().str.strip()
    # remove special characters
    dfi.columns = [re.sub(r'\W+', '_', col).strip('_') for col in dfi.columns]
    # drop empty cells
    dfi = dfi.dropna(how='all', axis=0)
    dfi['date'] = pd.to_datetime(dfi['period'], errors='coerce')
    dfi = dfi.dropna(subset='date')
    dfi.set_index('date', inplace=True)
    dfi = dfi.drop('period', axis=1)
    df0 = dfi.copy()
    
    # Government Operations – Revenues and Grants (Table F2)
    dfi = pd.read_excel(df, skiprows=4, header=[0,1,2], sheet_name=df.sheet_names[1])
    
    # column names
    dfi.columns = [' '.join(col).strip().lower() for col in dfi.columns.values]
    dfi.columns = dfi.columns.astype(str).str.lower().str.strip()
    dfi.columns = [re.sub(r'unnamed: \d+_level_\d+', '', str(col), flags=re.IGNORECASE).strip() for col in dfi.columns]
    dfi.columns = [re.sub(r'\W+', '_', col).strip('_') for col in dfi.columns]
    dfi.columns = [re.sub(r'\W+', '_', re.sub(r'\d+', '', str(col))).strip('_') for col in dfi.columns]
    dfi.columns = dfi.columns.astype(str).str.lower().str.strip()
    dfi = dfi.dropna(how='all', axis=0)
    
    # clean dates
    dfi = dfi.dropna(how='all', axis=0)
    dfi['date'] = pd.to_datetime(dfi['period'], errors='coerce')
    dfi = dfi.dropna(subset='date')
    dfi.set_index('date', inplace=True)
    dfi = dfi.drop('period', axis=1)
    
    # merge fiscal data
    df = pd.concat( [df0, dfi], axis=1 )
    df.index = df.index.to_period('M').to_timestamp()
    print(f'Data Range: {df.index.min()}-{df.index.max()}')

Retrieving historical: https://www.centralbank.org.bb/news/summary-of-government-operations
Data Range: 1995-03-01 00:00:00-2024-12-01 00:00:00


In [49]:
df0.tail(5)

,direct_tax_revenue,indirect_tax_revenue,non_tax_revenue_grants,current_expenditure,interest_component_of_current_expenditure,capital_expenditure,net_lending,total_revenues_grants,total_expenditure_net_lending,fiscal_balance,primary_balance
date,,,,,,,,,,,
2023-12-31,329.282222,458.024820,44.340903,752.172831,167.063477,39.962871,5.774822,831.647945,797.910524,33.737422,200.800899
2024-03-31,471.711744,501.067916,44.641288,992.885074,204.305867,255.835996,-8.519268,1017.420949,1240.201802,-222.780853,-18.474986
2024-06-30,584.489485,468.118831,66.157247,718.506192,167.100000,46.400052,4.333018,1118.765564,769.239261,349.526303,516.626303
2024-09-30,319.927631,429.186260,46.047713,867.343758,212.200000,68.116617,7.299341,795.161604,942.759716,-147.598112,64.601888
2024-12-31,477.859551,473.815664,44.615303,815.058084,170.066376,150.226360,6.462484,996.290518,971.746927,24.543591,194.609967


# Central Bank - Monetary Data

## International Reserves

In [50]:
url = "https://www.centralbank.org.bb/news/international-reserves"
r = requests.get(url=url, verify=None, headers=hdr).content

soup = BeautifulSoup(r)
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
links = [ link for link in links if "monetary" in link ]

url = links[0]
r = requests.get(url=url, verify=None, headers=hdr).content

soup = BeautifulSoup(r)
links = [ link.get("href") for link in soup('a') if link.get("href") is not None ]
links = [ link for link in links if "xlsx" in link ]

# ------------------------------------------------
# Retrieve excel
get = links[0]
content = requests.get(url=get, verify=False, headers=hdr).content
df = pd.ExcelFile(io.BytesIO(content))

# ---------------------------------------------------
# ---------------------------------------------------
# IRR

dfi = pd.read_excel(df, sheet_name='InternationalReserves', skiprows=2, header=[1,2,3])

# column names
dfi.columns = [' '.join(col).strip().lower() for col in dfi.columns.values]
dfi.columns = dfi.columns.astype(str).str.lower().str.strip()
dfi.columns = [re.sub(r'unnamed: \d+_level_\d+', '', str(col), flags=re.IGNORECASE).strip() for col in dfi.columns]
dfi.columns = [re.sub(r'\W+', '_', col).strip('_') for col in dfi.columns]
dfi.columns = [re.sub(r'\W+', '_', re.sub(r'\d+', '', str(col))).strip('_') for col in dfi.columns]
dfi.columns = dfi.columns.astype(str).str.lower().str.strip()
dfi = dfi.dropna(how='all', axis=0)

# Manage dates
dfi.columns.values[0] = "date"
dfi.drop([''], axis=1, inplace=True)
dfi.set_index('date', inplace=True)

# ---------------------------------------------------
# ---------------------------------------------------
# Monetary Base
df0 = pd.read_excel(df, sheet_name='NDA', skiprows=2, header=[1,2,3])

# column names
df0.columns = [' '.join(col).strip().lower() for col in df0.columns.values]
df0.columns = df0.columns.astype(str).str.lower().str.strip()
df0.columns = [re.sub(r'unnamed: \d+_level_\d+', '', str(col), flags=re.IGNORECASE).strip() for col in df0.columns]
df0.columns = [re.sub(r'\W+', '_', col).strip('_') for col in df0.columns]
df0.columns = [re.sub(r'\W+', '_', re.sub(r'\d+', '', str(col))).strip('_') for col in df0.columns]
df0.columns = df0.columns.astype(str).str.lower().str.strip()
df0 = df0.dropna(how='all', axis=0)

# Manage dates
df0.columns.values[0] = "date"
df0.drop([''], axis=1, inplace=True)
df0.set_index('date', inplace=True)
df0 = df0.dropna()

# ---------------------------------------------------
# ---------------------------------------------------
# Merge and Save
df = pd.concat([dfi, df0], axis=1)
df.to_csv(f'{PATH_RAW}/cbbb_monetary.csv')
print(f'Last updated: {df.index.max()}')
df.tail(5)

Last updated: 2026-02-01 00:00:00


,net_intl_reserves,gross_intl_reserves,gross_foreign_assets_total,gross_foreign_assets_gold_f_r_b_account,gross_foreign_assets_frn_currencies_in_hand,gross_foreign_assets_balances_held_abroad,gross_foreign_assets_foreign_securities_total,gross_foreign_assets_foreign_securities_us_treasury_bills,gross_foreign_assets_foreign_securities_us_securities_unencumbered,gross_foreign_assets_foreign_securities_other_government_securities,...,net_domestic_assets_net_claims_on_public_sector_net_credit_to_government,net_domestic_assets_net_claims_on_public_sector_deposit_liabilities_to_insurance_funds_and_levies,net_domestic_assets_net_claims_on_public_sector_deposit_liabilities_to_decentralised_agencies,net_domestic_assets_net_claims_on_public_sector_total,net_domestic_assets_credit_to_commercial_banks,net_domestic_assets_net_credit_to_rest_of_financial_system,net_domestic_assets_deposit_liabilities_to_non_monetary_intl_organisations,net_domestic_assets_sdr_allocations,net_domestic_assets_official_capital_and_reserves,net_domestic_assets_other
date,,,,,,,,,,,,,,,,,,,,,
2025-10-01,2.780590e+06,3.158497e+06,3.169098e+06,0.0,7967.18732,574278.00862,2.502157e+06,79214.8198,661412.18900,1.761530e+06,...,-275418.43829,0.0,-5229.31192,647585.50718,0.0,0.0,-4486.16690,-406419.47700,1.374692e+06,-257423.05986
2025-11-01,2.681518e+06,3.059408e+06,3.070598e+06,0.0,7233.59141,571669.04987,2.418865e+06,79214.8198,653171.91371,1.686478e+06,...,-362780.15831,0.0,-3380.99559,370964.70349,0.0,0.0,-4427.39984,-406419.47700,1.375948e+06,-132455.19018
2025-12-01,2.691609e+06,3.045727e+06,3.057582e+06,0.0,6494.85119,572586.39228,2.400156e+06,29708.0418,650424.31127,1.720023e+06,...,-334012.18298,0.0,-3589.90702,606703.76739,0.0,0.0,-3623.72056,-406419.47700,1.412966e+06,-207638.52761
2026-01-01,2.655666e+06,3.023690e+06,3.035265e+06,0.0,16424.21693,607480.01679,2.333385e+06,0.0000,634265.10258,1.699120e+06,...,-377918.11858,0.0,-4910.69379,523727.44502,0.0,0.0,-2625.43508,-426790.04367,1.404552e+06,-147116.48976
2026-02-01,2.652289e+06,3.020758e+06,3.033373e+06,0.0,10530.95161,607686.75888,2.347950e+06,0.0000,634259.60062,1.713691e+06,...,-377832.82357,0.0,-7218.18218,444502.85164,0.0,0.0,-1300.42982,-426790.04367,1.401917e+06,-154360.37372
